In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
import copy
import os
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# ==========================================
class TabularMLP_Weak(nn.Module):
    def __init__(self, input_dim):
        super(TabularMLP_Weak, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16), 
            nn.ReLU(),
            nn.Dropout(0.55), 
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.net(x)

class TabularOnlyDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.features)
    def __getitem__(self, idx): return self.features[idx], self.labels[idx]

# ==========================================
# 2. 计算三项指标的辅助函数
# ==========================================
def get_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return r2, rmse, mape

# ==========================================
# 3. 单次训练与全集合评估函数
# ==========================================
def run_single_experiment(seed, X_train, y_train_raw, X_val, y_val_raw, X_test, y_test_raw, 
                          y_train_norm, y_val_norm, y_test_norm, label_min, label_max, input_dim):
    torch.manual_seed(seed)
    np.random.seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    batch_size = 32
    num_epochs = 45 
    lr = 0.005
    wd = 0.02 

    train_loader = DataLoader(TabularOnlyDataset(X_train, y_train_norm), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TabularOnlyDataset(X_val, y_val_norm), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(TabularOnlyDataset(X_test, y_test_norm), batch_size=batch_size, shuffle=False)

    model = TabularMLP_Weak(input_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    best_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        for f, l in train_loader:
            f, l = f.to(device), l.to(device)
            optimizer.zero_grad()
            optimizer.step(None) # 占位
            loss = criterion(model(f), l)
            loss.backward()
            optimizer.step()
        
        model.eval()
        v_loss = 0
        with torch.no_grad():
            for f, l in val_loader:
                f, l = f.to(device), l.to(device)
                v_loss += criterion(model(f), l).item() * f.size(0)
        v_loss /= len(val_loader.dataset)
        
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_wts = copy.deepcopy(model.state_dict())

    # --- 全集合评估 ---
    model.load_state_dict(best_wts)
    model.eval()
    
    def predict_all(loader):
        preds = []
        with torch.no_grad():
            for f, _ in loader:
                p = model(f.to(device)).cpu().numpy().flatten()
                preds.extend(p)
        # 反归一化
        return np.array(preds) * (label_max - label_min) + label_min

    p_train = predict_all(DataLoader(TabularOnlyDataset(X_train, y_train_norm), batch_size=64, shuffle=False))
    p_val = predict_all(val_loader)
    p_test = predict_all(test_loader)
    res_train = get_metrics(y_train_raw, p_train)
    res_val = get_metrics(y_val_raw, p_val)
    res_test = get_metrics(y_test_raw, p_test)
    
    return res_train, res_val, res_test

# ==========================================
# 4. 执行 50 次循环并汇总表格
# ==========================================
def main():
    train_df = pd.read_csv('split_train_data.csv')
    val_df = pd.read_csv('split_val_data.csv')
    test_df = pd.read_csv('split_test_data.csv')

    train_num = train_df.select_dtypes(include=[np.number])
    val_num = val_df.select_dtypes(include=[np.number])
    test_num = test_df.select_dtypes(include=[np.number])

    X_tr_raw, y_tr_raw = train_num.iloc[:, :-1].values, train_num.iloc[:, -1].values
    X_va_raw, y_va_raw = val_num.iloc[:, :-1].values, val_num.iloc[:, -1].values
    X_te_raw, y_te_raw = test_num.iloc[:, :-1].values, test_num.iloc[:, -1].values

    label_min, label_max = y_tr_raw.min(), y_tr_raw.max()
    y_tr_n = (y_tr_raw - label_min) / (label_max - label_min)
    y_va_n = (y_va_raw - label_min) / (label_max - label_min)
    y_te_n = (y_te_raw - label_min) / (label_max - label_min)

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr_raw)
    X_va = scaler.transform(X_va_raw)
    X_te = scaler.transform(X_te_raw)

    input_dim = X_tr.shape[1]
    num_runs = 50
    
    # 统计容器
    stats = {
        'tr_r2': [], 'tr_rmse': [], 'tr_mape': [],
        'va_r2': [], 'va_rmse': [], 'va_mape': [],
        'te_r2': [], 'te_rmse': [], 'te_mape': []
    }

    print(f"🚀 开始执行 {num_runs} 次独立重复实验 (全集合评估)...")
    for i in range(num_runs):
        tr, va, te = run_single_experiment(i*7, X_tr, y_tr_raw, X_va, y_va_raw, X_te, y_te_raw, 
                                           y_tr_n, y_va_n, y_te_n, label_min, label_max, input_dim)
        
        stats['tr_r2'].append(tr[0]); stats['tr_rmse'].append(tr[1]); stats['tr_mape'].append(tr[2])
        stats['va_r2'].append(va[0]); stats['va_rmse'].append(va[1]); stats['va_mape'].append(va[2])
        stats['te_r2'].append(te[0]); stats['te_rmse'].append(te[1]); stats['te_mape'].append(te[2])
        
        if (i+1) % 10 == 0:
            print(f"Run {i+1}/50: Test R2 = {te[0]:.4f}")

    # 生成最终汇总表格
    def get_mean_std(key):
        return f"{np.mean(stats[key]):.4f} ± {np.std(stats[key]):.4f}"

    final_results = [{
        'Model': 'Weakened MLP',
        'Train R2': get_mean_std('tr_r2'), 'Train RMSE': get_mean_std('tr_rmse'), 'Train MAPE': get_mean_std('tr_mape'),
        'Val R2': get_mean_std('va_r2'),   'Val RMSE': get_mean_std('va_rmse'),   'Val MAPE': get_mean_std('va_mape'),
        'Test R2': get_mean_std('te_r2'),  'Test RMSE': get_mean_std('te_rmse'),  'Test MAPE': get_mean_std('te_mape')
    }]

    df_final = pd.DataFrame(final_results)
    
    print("\n" + "="*120)
    print(" "*45 + "神经网络全集合 50 次实验汇总统计")
    print("="*120)
    print(df_final.to_string(index=False))
    print("="*120)

    # 保存结果
    df_final.to_csv('MLP_Comprehensive_50Runs_Results.csv', index=False)
    print(f"✅ 完整评估表格已保存至: MLP_Comprehensive_50Runs_Results.csv")

if __name__ == '__main__':
    main()

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


🚀 开始执行 50 次独立重复实验 (全集合评估)...
Run 10/50: Test R2 = -0.4155
Run 20/50: Test R2 = 0.5427
Run 30/50: Test R2 = 0.4757
Run 40/50: Test R2 = 0.5516
Run 50/50: Test R2 = 0.2364

                                             神经网络全集合 50 次实验汇总统计
       Model        Train R2      Train RMSE      Train MAPE          Val R2        Val RMSE        Val MAPE         Test R2       Test RMSE       Test MAPE
Weakened MLP 0.4394 ± 0.1954 0.0514 ± 0.0081 0.0103 ± 0.0014 0.4379 ± 0.0524 0.0556 ± 0.0026 0.0117 ± 0.0007 0.4555 ± 0.2111 0.0578 ± 0.0099 0.0124 ± 0.0016
✅ 完整评估表格已保存至: MLP_Comprehensive_50Runs_Results.csv
